# step B — pool-500 (RQ2 관측, POOL 50×50 + 블록 커버리지)

**대응 RQ:** RQ2 관측 — 이름을 생성하는 **바로 그 순간** 모델이 어디를 보고(어텐션) 무엇을 실어 나르는지(‖v‖·‖av‖)를 내부에서 본다.

**두 관측(view)** (측정 설계는 파일럿과 동일):
- **View 1 — flip 2×2:** 선행 camel 6 / snake 6 균형, 지침만 camel↔snake. '더 보는 쪽'이 **위반 표기를 따라 뒤집히는가**.
- **View 2 — 위반 개수 스윕:** 지침 camel 고정, 준수 개수 4→0. **지침 구간 어텐션이 평탄한가**(실패=지침 결핍 아님).

**파일럿 대비 (POOL 확장):** 이름 창고 80→**504(50×50)**. 파일럿은 seed로 12개 랜덤 표집이었으나,
scaleup은 **블록**으로 서로 다른 이름 500+개를 덮는다(seed는 위치·문맥 변주). 다양성은 seed가 아니라 **50×50 풀**에서.

**코드 동일 보장(재현성):** 측정은 파일럿과 **똑같은 `run(..., mode='observe')`**. 단 POOL은 표집이라 파일럿(80풀)과 글자 그대로 재현되진 않음 — 새 500 샘플로 진행(파일럿은 `results/stepB/` 불변 보존). 원래 이름 80개는 새 풀의 부분집합.

설계 문서: `docs/stepB/scaleup-500.md` (파일럿: `docs/stepB/plan.md`).

> **메모리(무료 T4):** 어텐션 관측은 `output_attentions`로 전체 어텐션 행렬을 떠서 메모리를 쓴다. stepB 합성 프롬프트(수백 토큰)는 T4에서 감당. **eager 어텐션 필수**(`load_model(attn_implementation='eager')`).
> **시간·규모:** 기본 7 spec × 42 블록 × 1 seed = **294 조건**(생성 아님, 1-forward라 stepA보다 가벼움). **재개 가능.** 무료 티어면 `BLOCKS`를 줄이거나, 위치 강건성을 원하면 `SEEDS`를 [0,1,2]로.

In [ ]:
# 환경 설정 — 설치, GPU 확인, 시드 고정
!pip install -q transformers accelerate torch matplotlib pandas numpy

import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (주의: 매우 느림)')

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('seed fixed:', SEED)

In [ ]:
# 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin stepB/pool-500
!git checkout stepB/pool-500
!git pull --quiet origin stepB/pool-500
!pip install -e . -q
import sys; sys.path.insert(0, 'src')

In [ ]:
# 조건 설정 — View1 flip + View2 스윕, 이름은 블록으로 500+ 커버
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation)
from harness.tasks import NAME_PAIR_POOL

MODEL = ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct', family='qwen', dtype='float16')
REF_FRAC = 0.7                                  # 요약 기준 층의 상대 위치
N_FUNCTIONS = 12
N_BLOCKS = len(NAME_PAIR_POOL) // N_FUNCTIONS   # 504//12 = 42 블록(500+ 이름 커버)
BLOCKS = list(range(N_BLOCKS))
SEEDS = [0]                                     # 블록이 다양성 제공. 위치 강건성 원하면 [0,1,2]

# (목표 표기, 선행 camel 개수) — 두 view 합쳐 중복 제거
FLIP  = [(Notation.CAMEL, 6), (Notation.SNAKE, 6)]        # View1: 6/6 균형, 지침 flip
SWEEP = [(Notation.CAMEL, n) for n in (4, 3, 2, 1, 0)]     # View2: 지침 camel, 위반↑ (stepA 동일)
SPECS = list(dict.fromkeys(FLIP + SWEEP))                  # (camel,6) 공유

def make(target, n, block, s):
    return Condition(
        model=MODEL,
        preceding=PrecedingCode(n_compliant=n, n_functions=N_FUNCTIONS,
                                composition=Composition.POOL, pool_block=block),
        instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=target),
        seed=s,
    )

conditions = [make(t, n, b, s) for (t, n) in SPECS for b in BLOCKS for s in SEEDS]

# 실행 전 예측 (결과와 함께 보존, CLAUDE.md §6)
PREDICTION = ('View1: 충돌 표기 토큰을 더 봄(camel지침->snake, snake지침->camel), ‖v‖ 평탄. '
              'View2: 위반이 늘어도 지침 구간 어텐션은 평탄(실패=지침 결핍 아님). '
              '500개 이름(블록)에서 효과가 특정 단어에 묶이지 않음.')
print(f'풀 {len(NAME_PAIR_POOL)}개 이름, {N_BLOCKS}블록')
print(f'{len(conditions)} 조건 = {len(SPECS)} spec x {len(BLOCKS)} block x {len(SEEDS)} seed')
print('specs (target, n_compliant):', [(t.value, n) for (t, n) in SPECS])

In [ ]:
# 실행 — 조건별 관측 + 즉시 저장(재개) + 중간 누적 요약. 로직은 harness가 수행(파일럿과 동일 observe).
from collections import defaultdict
from harness import run, ResultRecord, save_result, result_path
from harness.results import load_result
from harness.model import load_model

STEP = 'stepB_scaleup500'
handle = load_model(MODEL, attn_implementation='eager')   # 어텐션 가중치 읽기 위해 eager 필수
REF_LAYER = int(handle.num_layers * REF_FRAC)
print('layers:', handle.num_layers, '| GQA:', handle.gqa_info(), '| 기준층 L%d' % REF_LAYER)

acc = defaultdict(lambda: {'camel': [], 'snake': []})    # View1 누적(6/6): 지침별 토큰 어텐션
new = skipped = 0
for i, c in enumerate(conditions, 1):
    p = result_path(c, step=STEP)
    if p.exists():
        rec = load_result(p); skipped += 1
    else:
        out = run(c, handle=handle, mode='observe')
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step=STEP, rq='RQ2', prediction=PREDICTION))
        rec = load_result(p); new += 1
    pl = rec.metrics.per_layer.get(REF_LAYER, {})
    n = rec.condition.preceding.n_compliant
    tgt = rec.condition.instruction.target_notation.value
    if n == 6 and 'code_camel__attention_weight' in pl and 'code_snake__attention_weight' in pl:
        acc[tgt]['camel'].append(pl['code_camel__attention_weight'])
        acc[tgt]['snake'].append(pl['code_snake__attention_weight'])
    if i % 20 == 0 or i == len(conditions):              # 중간 누적 출력
        print(f'[{i}/{len(conditions)}] 새 {new} / 건너뜀 {skipped}  (기준 L{REF_LAYER})')
        for t in ('camel', 'snake'):                     # View1: 충돌 표기를 더 보나(누적)
            cs, ss = acc[t]['camel'], acc[t]['snake']
            if cs:
                mc, ms = sum(cs)/len(cs), sum(ss)/len(ss)
                more = 'snake' if ms > mc else 'camel'
                print(f'    [View1] 지침={t}: camel토큰 {mc:.4f} vs snake토큰 {ms:.4f} '
                      f'-> {more} 더 봄 (누적 n={len(cs)}, 이름 {len(cs)*6}개)')
print(f'완료: 새로 {new}, 건너뜀 {skipped}, 총 {len(conditions)}')

In [ ]:
# 결과 로드 — results/stepB_scaleup500/ 에 불변 저장된 이 실험 조건들을 모은다
from harness import result_path
from harness.results import load_result

records = [load_result(result_path(c, step=STEP)) for c in conditions]
print('로드:', len(records), '건 -> results/'+STEP+'/')

In [ ]:
# 요약 — View1(flip 2x2) + View2(위반 개수 스윕). 500개 이름 전체 집계.
import pandas as pd, numpy as np, matplotlib.pyplot as plt

rows = []
for r in records:
    t = r.condition.instruction.target_notation.value
    n = r.condition.preceding.n_compliant
    for layer, pl in r.metrics.per_layer.items():
        rows.append({'target': t, 'n_compliant': n, 'layer': int(layer),
                     'camel_attn': pl.get('code_camel__attention_weight'),
                     'snake_attn': pl.get('code_snake__attention_weight'),
                     'instr_attn': pl.get('instruction__attention_weight'),
                     'camel_av': pl.get('code_camel__av_norm'),
                     'snake_av': pl.get('code_snake__av_norm'),
                     'camel_v': pl.get('code_camel__v_norm'),
                     'snake_v': pl.get('code_snake__v_norm')})
df = pd.DataFrame(rows)
REF_LAYER = int((df.layer.max() + 1) * REF_FRAC)

# ── View 1 — flip 2x2 (선행 6/6) : 충돌 표기를 더 보는가 ──
bal = df[(df.n_compliant == 6) & (df.layer == REF_LAYER)]
tab = bal.groupby('target')[['camel_attn', 'snake_attn']].mean()
tab['더_본_쪽'] = np.where(tab['snake_attn'] > tab['camel_attn'], 'snake', 'camel')
print(f'[View1] flip 2x2 어텐션 @L{REF_LAYER} (충돌 표기를 더 보는가, 이름 500 집계):'); print(tab.round(4))
print('  지침 구간 어텐션 (조건 간 평탄 기대):')
print(bal.groupby('target')['instr_attn'].mean().round(4))
print('  ‖v‖ (크기, 평탄 기대):')
print(bal.groupby('target')[['camel_v', 'snake_v']].mean().round(3))

# ── View 2 — 위반 개수 스윕 (지침=camel 고정) : 지침 어텐션 평탄한가 ──
sw = df[(df.target == 'camel') & (df.layer == REF_LAYER)].groupby('n_compliant')
sweep_tab = pd.DataFrame({'지침_어텐션': sw['instr_attn'].mean(),
                          'snake(충돌)_av합': sw['snake_av'].mean()}).sort_index(ascending=False)
print(f'\n[View2] 위반 개수 스윕 @L{REF_LAYER} (n_compliant 6->0, 위반↑):'); print(sweep_tab.round(4))

# ── 플롯 ──
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
g = df[(df.n_compliant == 6) & (df.target == 'camel')].groupby('layer')[['camel_attn', 'snake_attn']].mean()
ax[0].plot(g.index, g['camel_attn'], marker='.', label='camel 토큰(준수)')
ax[0].plot(g.index, g['snake_attn'], marker='.', label='snake 토큰(충돌)')
ax[0].axvline(REF_LAYER, color='gray', ls='--', alpha=.5)
ax[0].set_title('View1: camel 지침, 층별 토큰 어텐션'); ax[0].set_xlabel('layer')
ax[0].set_ylabel('구간 어텐션 합(평균)'); ax[0].grid(True, alpha=.3); ax[0].legend()
xs = sorted(df.n_compliant.unique(), reverse=True)
inst_by_n = [df[(df.target=='camel') & (df.n_compliant==n) & (df.layer==REF_LAYER)]['instr_attn'].mean() for n in xs]
ax[1].plot([str(n) for n in xs], inst_by_n, marker='o', color='C2')
ax[1].set_title('View2: 위반↑ 에도 지침 어텐션 평탄?'); ax[1].set_xlabel('n_compliant (6->0)')
ax[1].set_ylabel('지침 구간 어텐션'); ax[1].grid(True, alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# 밑줄/형태 마커 분석 — 형태 신호가 '어근 단어'가 아니라 '표층 마커'(_, 대문자)에 실리나.
# snake 표식=밑줄 '_', camel 표식=내부 대문자. 마커 토큰 av가 어근보다 크면 형태를 읽는다는 직접 증거(RQ2).
REF = str(REF_LAYER)
rows = []
for r in records:
    td = r.metrics.extra.get('token_detail', {})
    checks = [('code_snake', lambda s: '_' in s, '밑줄'),
              ('code_camel', lambda s: any(ch.isupper() for ch in s), '대문자')]
    for sp, is_marker, mlabel in checks:
        d = td.get(sp)
        if not d or not d.get('tokens'):
            continue
        arr = d['per_layer'].get(REF)
        if not arr:
            continue
        for tok, a, av in zip(d['tokens'], arr['a'], arr['av']):
            rows.append({'span': sp, 'text': tok['text'], 'a': a, 'av': av,
                         'kind': mlabel if is_marker(tok['text']) else '어근'})
det = pd.DataFrame(rows)
for sp in ['code_snake', 'code_camel']:
    sub = det[det.span == sp] if not det.empty else det
    if sub.empty:
        continue
    print(f'[{sp}] 형태 마커 vs 어근 토큰 (@L{REF}):')
    print(sub.groupby('kind')[['a', 'av']].mean().round(5))
    top = sub.sort_values('av', ascending=False)['text'].head(6).tolist()
    print('  av 상위 토큰:', top, '\n')

In [ ]:
# 결과 다운로드 — results/stepB_scaleup500 을 zip으로 묶어 내려받는다
import shutil
shutil.make_archive('stepB_scaleup500_results', 'zip', 'results/'+STEP)
try:
    from google.colab import files
    files.download('stepB_scaleup500_results.zip')
except Exception as e:
    print('Colab 아님(수동 다운로드): stepB_scaleup500_results.zip', e)